
# Roxy notebook example: Quasi-sequence-order descriptors

This notebook is a **reference implementation example** for the **quasi-sequence-order descriptor family** in Roxy.

Quasi-sequence-order descriptors extend composition-based representations by incorporating **sequence-order coupling information** derived from residue-level physicochemical distances.

They are closely related to PseAAC, but conceptually emphasize:

- amino acid composition
- pairwise coupling between residues at different lags
- sequence-order effects summarized through distance-based correlation terms

## Covered outputs

This notebook implements:

- amino acid composition baseline
- sequence-order coupling factors for configurable lags
- quasi-sequence-order descriptors using multiple physicochemical properties
- configurable `lambda` (number of lags)
- configurable weight parameter `w`
- normalized quasi-sequence-order vectors
- a simplified single-property variant
- class-style implementation for later migration into Roxy

The notebook is written as a **clean teaching implementation** so it can later become part of the real Roxy package.


In [1]:

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "qso_1",
            "qso_2",
            "qso_3",
            "qso_4",
            "qso_5",
            "qso_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,qso_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,qso_2,GGGGGGGGGGGGGGG,B
2,qso_3,KRRKRRKRRKRRDDDDEE,A
3,qso_4,ACDEFGHIKLMNPQRSTVWY,B
4,qso_5,PPPPGSSSSSTTTTNNQQQ,A
5,qso_6,MSTNPKPQRITLKDGNKVELV,B


## Constants and residue-level properties

In [3]:

STANDARD_AA = list("ACDEFGHIKLMNPQRSTVWY")
STANDARD_AA_SET = set(STANDARD_AA)

HYDROPHOBICITY = {
    "A": 0.62, "C": 0.29, "D": -0.90, "E": -0.74, "F": 1.19,
    "G": 0.48, "H": -0.40, "I": 1.38, "K": -1.50, "L": 1.06,
    "M": 0.64, "N": -0.78, "P": 0.12, "Q": -0.85, "R": -2.53,
    "S": -0.18, "T": -0.05, "V": 1.08, "W": 0.81, "Y": 0.26,
}

HYDROPHILICITY = {
    "A": -0.50, "C": -1.00, "D": 3.00, "E": 3.00, "F": -2.50,
    "G": 0.00, "H": -0.50, "I": -1.80, "K": 3.00, "L": -1.80,
    "M": -1.30, "N": 0.20, "P": 0.00, "Q": 0.20, "R": 3.00,
    "S": 0.30, "T": -0.40, "V": -1.50, "W": -3.40, "Y": -2.30,
}

SIDECHAIN_MASS = {
    "A": 15.0, "C": 47.0, "D": 59.0, "E": 73.0, "F": 91.0,
    "G": 1.0, "H": 82.0, "I": 57.0, "K": 72.0, "L": 57.0,
    "M": 75.0, "N": 58.0, "P": 41.0, "Q": 72.0, "R": 100.0,
    "S": 31.0, "T": 45.0, "V": 43.0, "W": 130.0, "Y": 107.0,
}

QSO_PROPERTIES = {
    "hydrophobicity": HYDROPHOBICITY,
    "hydrophilicity": HYDROPHILICITY,
    "sidechain_mass": SIDECHAIN_MASS,
}


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA_SET])


def zscore_scale(scale: dict) -> dict:
    values = np.array([scale[aa] for aa in STANDARD_AA], dtype=float)
    mean = values.mean()
    std = values.std(ddof=0)
    return {aa: (scale[aa] - mean) / std for aa in STANDARD_AA}


def aac_frequencies(seq: str) -> dict:
    seq = clean_sequence(seq)
    if len(seq) == 0:
        return {aa: np.nan for aa in STANDARD_AA}
    return {aa: seq.count(aa) / len(seq) for aa in STANDARD_AA}


def residue_distance(aa1: str, aa2: str, normalized_scales: dict) -> float:
    diffs = []
    for _, scale in normalized_scales.items():
        diffs.append((scale[aa1] - scale[aa2]) ** 2)
    return float(np.mean(diffs))


def qso_coupling_factor(seq: str, lag: int, normalized_scales: dict) -> float:
    seq = clean_sequence(seq)
    n = len(seq)
    if n <= lag or lag < 1:
        return np.nan

    values = []
    for i in range(n - lag):
        aa1 = seq[i]
        aa2 = seq[i + lag]
        values.append(residue_distance(aa1, aa2, normalized_scales))

    return float(np.mean(values)) if len(values) > 0 else np.nan


def qso_coupling_factor_single_property(seq: str, lag: int, normalized_scale: dict) -> float:
    seq = clean_sequence(seq)
    n = len(seq)
    if n <= lag or lag < 1:
        return np.nan

    values = []
    for i in range(n - lag):
        aa1 = seq[i]
        aa2 = seq[i + lag]
        values.append((normalized_scale[aa1] - normalized_scale[aa2]) ** 2)

    return float(np.mean(values)) if len(values) > 0 else np.nan


## Core quasi-sequence-order implementation

In [5]:

def quasi_sequence_order(seq: str, lam: int = 5, w: float = 0.1, properties=None) -> dict:
    """Compute quasi-sequence-order descriptors using multiple physicochemical properties."""
    seq = clean_sequence(seq)

    if properties is None:
        properties = ("hydrophobicity", "hydrophilicity", "sidechain_mass")

    out = {
        "qso_length": len(seq),
        "qso_valid_residue_count": len(seq),
        "qso_lambda": lam,
        "qso_weight": w,
    }

    if len(seq) == 0:
        return out

    aac = aac_frequencies(seq)
    normalized_scales = {
        prop: zscore_scale(QSO_PROPERTIES[prop]) for prop in properties
    }

    couplings = []
    for lag in range(1, lam + 1):
        cf = qso_coupling_factor(seq, lag, normalized_scales)
        couplings.append(cf if not np.isnan(cf) else 0.0)

    denominator = 1.0 + w * sum(couplings)

    # Composition part
    for aa in STANDARD_AA:
        out[f"qso_{aa}"] = aac[aa] / denominator

    # Sequence-order coupling part
    for i, cf in enumerate(couplings, start=1):
        out[f"qso_tau_{i}"] = (w * cf) / denominator

    out["qso_feature_sum"] = sum(out[f"qso_{aa}"] for aa in STANDARD_AA) + sum(
        out[f"qso_tau_{i}"] for i in range(1, lam + 1)
    )

    return out


## Simplified single-property variant

In [6]:

def single_property_qso(seq: str, lam: int = 5, w: float = 0.1, property_name: str = "hydrophobicity") -> dict:
    seq = clean_sequence(seq)

    out = {
        "sqso_length": len(seq),
        "sqso_valid_residue_count": len(seq),
        "sqso_lambda": lam,
        "sqso_weight": w,
        "sqso_property": property_name,
    }

    if len(seq) == 0:
        return out

    aac = aac_frequencies(seq)
    normalized_scale = zscore_scale(QSO_PROPERTIES[property_name])

    couplings = []
    for lag in range(1, lam + 1):
        cf = qso_coupling_factor_single_property(seq, lag, normalized_scale)
        couplings.append(cf if not np.isnan(cf) else 0.0)

    denominator = 1.0 + w * sum(couplings)

    for aa in STANDARD_AA:
        out[f"sqso_{aa}"] = aac[aa] / denominator

    for i, cf in enumerate(couplings, start=1):
        out[f"sqso_tau_{i}"] = (w * cf) / denominator

    out["sqso_feature_sum"] = sum(out[f"sqso_{aa}"] for aa in STANDARD_AA) + sum(
        out[f"sqso_tau_{i}"] for i in range(1, lam + 1)
    )

    return out


## Functional usage on one sequence

In [7]:

example = quasi_sequence_order(df_demo.loc[0, "sequence"], lam=5, w=0.1)
list(example.items())[:18]


[('qso_length', 24),
 ('qso_valid_residue_count', 24),
 ('qso_lambda', 5),
 ('qso_weight', 0.1),
 ('qso_A', 0.02003523922497865),
 ('qso_C', 0.0),
 ('qso_D', 0.0),
 ('qso_E', 0.0),
 ('qso_F', 0.0801409568999146),
 ('qso_G', 0.02003523922497865),
 ('qso_H', 0.0),
 ('qso_I', 0.02003523922497865),
 ('qso_K', 0.02003523922497865),
 ('qso_L', 0.06010571767493595),
 ('qso_M', 0.02003523922497865),
 ('qso_N', 0.0),
 ('qso_P', 0.0),
 ('qso_Q', 0.0)]

## Apply quasi-sequence-order descriptors to the full dataset

In [8]:

df_qso = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(lambda x: quasi_sequence_order(x, lam=5, w=0.1)).apply(pd.Series),
    ],
    axis=1,
)

df_qso.head()


,sequence_id,sequence,label,qso_length,qso_valid_residue_count,qso_lambda,qso_weight,qso_A,qso_C,qso_D,...,qso_T,qso_V,qso_W,qso_Y,qso_tau_1,qso_tau_2,qso_tau_3,qso_tau_4,qso_tau_5,qso_feature_sum
0,qso_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,5.0,0.1,0.020035,0.00000,0.00000,...,0.020035,0.04007,0.020035,0.020035,0.107844,0.123589,0.120323,0.083052,0.084346,1.0
1,qso_2,GGGGGGGGGGGGGGG,B,15.0,15.0,5.0,0.1,0.000000,0.00000,0.00000,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
2,qso_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,5.0,0.1,0.000000,0.00000,0.17883,...,0.000000,0.00000,0.000000,0.000000,0.029644,0.039711,0.018537,0.047788,0.059586,1.0
3,qso_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,5.0,0.1,0.024210,0.02421,0.02421,...,0.024210,0.02421,0.024210,0.024210,0.095731,0.106266,0.099119,0.124234,0.090453,1.0
4,qso_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,5.0,0.1,0.000000,0.00000,0.00000,...,0.193334,0.00000,0.000000,0.000000,0.008214,0.011716,0.016631,0.021750,0.023355,1.0


## Apply single-property QSO variant

In [9]:

df_sqso = pd.concat(
    [
        df_demo[["sequence_id", "sequence"]],
        df_demo["sequence"].apply(
            lambda x: single_property_qso(
                x,
                lam=5,
                w=0.1,
                property_name="hydrophobicity",
            )
        ).apply(pd.Series),
    ],
    axis=1,
)

df_sqso.head()


,sequence_id,sequence,sqso_length,sqso_valid_residue_count,sqso_lambda,sqso_weight,sqso_property,sqso_A,sqso_C,sqso_D,...,sqso_T,sqso_V,sqso_W,sqso_Y,sqso_tau_1,sqso_tau_2,sqso_tau_3,sqso_tau_4,sqso_tau_5,sqso_feature_sum
0,qso_1,MKWVTFISLLFLFSSAYSRGVFRR,24,24,5,0.1,hydrophobicity,0.019166,0.000000,0.000000,...,0.019166,0.038331,0.019166,0.019166,0.102134,0.146488,0.129134,0.082564,0.079705,1.0
1,qso_2,GGGGGGGGGGGGGGG,15,15,5,0.1,hydrophobicity,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
2,qso_3,KRRKRRKRRKRRDDDDEE,18,18,5,0.1,hydrophobicity,0.000000,0.000000,0.155773,...,0.000000,0.000000,0.000000,0.000000,0.043896,0.059016,0.028175,0.072167,0.095767,1.0
3,qso_4,ACDEFGHIKLMNPQRSTVWY,20,20,5,0.1,hydrophobicity,0.022594,0.022594,0.022594,...,0.022594,0.022594,0.022594,0.022594,0.096499,0.102420,0.091924,0.149336,0.107940,1.0
4,qso_5,PPPPGSSSSSTTTTNNQQQ,19,19,5,0.1,hydrophobicity,0.000000,0.000000,0.000000,...,0.194121,0.000000,0.000000,0.000000,0.006042,0.010079,0.015242,0.021062,0.025498,1.0


## Inspect QSO columns

In [10]:

qso_cols = [c for c in df_qso.columns if c.startswith("qso_") and c not in {"qso_length", "qso_valid_residue_count", "qso_lambda", "qso_weight"}]
len(qso_cols), qso_cols[:15]


(26,
 ['qso_A',
  'qso_C',
  'qso_D',
  'qso_E',
  'qso_F',
  'qso_G',
  'qso_H',
  'qso_I',
  'qso_K',
  'qso_L',
  'qso_M',
  'qso_N',
  'qso_P',
  'qso_Q',
  'qso_R'])

In [11]:

df_qso[
    [
        "sequence_id",
        "qso_A",
        "qso_C",
        "qso_D",
        "qso_tau_1",
        "qso_tau_2",
        "qso_tau_3",
        "qso_feature_sum",
    ]
]


,sequence_id,qso_A,qso_C,qso_D,qso_tau_1,qso_tau_2,qso_tau_3,qso_feature_sum
0,qso_1,0.020035,0.00000,0.000000,0.107844,0.123589,0.120323,1.0
1,qso_2,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,1.0
2,qso_3,0.000000,0.00000,0.178830,0.029644,0.039711,0.018537,1.0
3,qso_4,0.024210,0.02421,0.024210,0.095731,0.106266,0.099119,1.0
4,qso_5,0.000000,0.00000,0.000000,0.008214,0.011716,0.016631,1.0
5,qso_6,0.000000,0.00000,0.024864,0.105868,0.093208,0.085753,1.0


## Dataset-level summary

In [12]:

qso_summary = (
    df_qso[qso_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

qso_summary.head(15)


,descriptor,mean_value
0,qso_feature_sum,1.000000
1,qso_G,0.186240
2,qso_R,0.077807
3,qso_tau_4,0.063604
4,qso_tau_2,0.062415
5,qso_S,0.061814
6,qso_tau_1,0.057883
7,qso_tau_5,0.057661
8,qso_tau_3,0.056727
9,qso_K,0.049611


## Sanity checks

In [13]:

assert "qso_A" in df_qso.columns
assert "qso_tau_1" in df_qso.columns
assert "qso_tau_5" in df_qso.columns
assert "qso_feature_sum" in df_qso.columns
assert df_qso["qso_length"].min() > 0

assert np.allclose(df_qso["qso_feature_sum"], 1.0)
assert np.allclose(df_sqso["sqso_feature_sum"], 1.0)

print(f"Number of QSO descriptor columns: {len(qso_cols)}")
print("Quasi-sequence-order descriptor checks passed.")


Number of QSO descriptor columns: 26
Quasi-sequence-order descriptor checks passed.


## Class-style implementation closer to the real package

In [14]:

class QuasiSequenceOrderDescriptors:
    """Example class-style quasi-sequence-order implementation for later migration into Roxy."""

    def __init__(self, lam: int = 5, w: float = 0.1, properties=None):
        self.lam = lam
        self.w = w
        self.properties = properties

    def transform_sequence(self, seq: str) -> dict:
        return quasi_sequence_order(
            seq,
            lam=self.lam,
            w=self.w,
            properties=self.properties,
        )

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


qso_transformer = QuasiSequenceOrderDescriptors(lam=5, w=0.1)
qso_matrix = qso_transformer.transform(df_demo["sequence"].tolist())
qso_matrix.head()


,qso_length,qso_valid_residue_count,qso_lambda,qso_weight,qso_A,qso_C,qso_D,qso_E,qso_F,qso_G,...,qso_T,qso_V,qso_W,qso_Y,qso_tau_1,qso_tau_2,qso_tau_3,qso_tau_4,qso_tau_5,qso_feature_sum
0,24,24,5,0.1,0.020035,0.00000,0.00000,0.000000,0.080141,0.020035,...,0.020035,0.04007,0.020035,0.020035,0.107844,0.123589,0.120323,0.083052,0.084346,1.0
1,15,15,5,0.1,0.000000,0.00000,0.00000,0.000000,0.000000,1.000000,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
2,18,18,5,0.1,0.000000,0.00000,0.17883,0.089415,0.000000,0.000000,...,0.000000,0.00000,0.000000,0.000000,0.029644,0.039711,0.018537,0.047788,0.059586,1.0
3,20,20,5,0.1,0.024210,0.02421,0.02421,0.024210,0.024210,0.024210,...,0.024210,0.02421,0.024210,0.024210,0.095731,0.106266,0.099119,0.124234,0.090453,1.0
4,19,19,5,0.1,0.000000,0.00000,0.00000,0.000000,0.000000,0.048333,...,0.193334,0.00000,0.000000,0.000000,0.008214,0.011716,0.016631,0.021750,0.023355,1.0


## Merge transformer output back to the dataset

In [15]:

df_qso_class = pd.concat([df_demo, qso_matrix], axis=1)
df_qso_class.head()


,sequence_id,sequence,label,qso_length,qso_valid_residue_count,qso_lambda,qso_weight,qso_A,qso_C,qso_D,...,qso_T,qso_V,qso_W,qso_Y,qso_tau_1,qso_tau_2,qso_tau_3,qso_tau_4,qso_tau_5,qso_feature_sum
0,qso_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,5,0.1,0.020035,0.00000,0.00000,...,0.020035,0.04007,0.020035,0.020035,0.107844,0.123589,0.120323,0.083052,0.084346,1.0
1,qso_2,GGGGGGGGGGGGGGG,B,15,15,5,0.1,0.000000,0.00000,0.00000,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0
2,qso_3,KRRKRRKRRKRRDDDDEE,A,18,18,5,0.1,0.000000,0.00000,0.17883,...,0.000000,0.00000,0.000000,0.000000,0.029644,0.039711,0.018537,0.047788,0.059586,1.0
3,qso_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,5,0.1,0.024210,0.02421,0.02421,...,0.024210,0.02421,0.024210,0.024210,0.095731,0.106266,0.099119,0.124234,0.090453,1.0
4,qso_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,5,0.1,0.000000,0.00000,0.00000,...,0.193334,0.00000,0.000000,0.000000,0.008214,0.011716,0.016631,0.021750,0.023355,1.0



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move physicochemical property scales into `roxy/core/constants.py`
- move helper logic into `roxy/sequence/qso.py`
- expose a class such as `QuasiSequenceOrderDescriptors`
- allow configurable:
  - lambda
  - weight parameter
  - selected physicochemical properties
  - multi-property vs single-property variants
- add tests for:
  - empty sequences
  - short sequences where lambda exceeds meaningful order depth
  - lower-case input
  - invalid characters removed during cleaning
  - descriptor vector normalization to 1.0


## Optional export

In [16]:
# df_qso.to_csv("demo_qso_descriptors.csv", index=False)
